**Step 1: Install packages**

In [ ]:
!pip install -q -U langgraph langgraph-checkpoint-sqlite langchain langchain-openai pydantic

**Step 2: Add your API key**

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

**Step 3: Imports**

In [ ]:
import json
import sqlite3
from typing import TypedDict, Literal, Optional, List, Dict, Any

from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.sqlite import SqliteSaver

**Step 4: Create the LLM**

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

**Step 5: Create a small knowledge base**

In [ ]:
KNOWLEDGE_BASE = [
    {
        "id": "kb_001",
        "category": "billing",
        "title": "Refund policy",
        "content": (
            "Customers can request a refund within 7 days of purchase for duplicate charges "
            "or accidental purchases. Refunds for subscription renewals require account verification."
        ),
    },
    {
        "id": "kb_002",
        "category": "billing",
        "title": "Invoice and payment issues",
        "content": (
            "If a payment fails, ask the customer to verify card details, billing address, "
            "and available balance. Escalate repeated charge failures after 2 attempts."
        ),
    },
    {
        "id": "kb_003",
        "category": "technical",
        "title": "App crashes on login",
        "content": (
            "If the app crashes on login, ask the user to clear cache, update to the latest version, "
            "restart the device, and try again. Escalate if crash logs are available or issue persists."
        ),
    },
    {
        "id": "kb_004",
        "category": "technical",
        "title": "Password reset troubleshooting",
        "content": (
            "If password reset email is not received, ask the customer to check spam, confirm the email address, "
            "and retry after 5 minutes. Escalate if the account is locked."
        ),
    },
    {
        "id": "kb_005",
        "category": "order",
        "title": "Order status policy",
        "content": (
            "If an order is marked shipped, provide the tracking window of 24 hours. "
            "If tracking is unavailable after 24 hours, escalate to logistics support."
        ),
    },
    {
        "id": "kb_006",
        "category": "account",
        "title": "Account locked procedure",
        "content": (
            "For locked accounts, verify the registered email, last successful login, and customer ID. "
            "Do not unlock manually without verification. Route high-risk account access issues for review."
        ),
    },
]

**Step 6: Define structured outputs**

In [ ]:
class TicketClassification(BaseModel):
    category: Literal["billing", "technical", "order", "account", "general"] = Field(
        description="Main category of the support ticket"
    )
    intent: Literal[
        "refund_request",
        "payment_issue",
        "bug_report",
        "password_reset",
        "order_status",
        "account_access",
        "complaint",
        "general_question",
        "other",
    ] = Field(description="Customer intent")
    priority: Literal["low", "medium", "high", "urgent"] = Field(
        description="Urgency of the ticket"
    )
    sentiment: Literal["positive", "neutral", "negative", "frustrated"] = Field(
        description="Customer sentiment"
    )
    confidence: float = Field(description="Confidence from 0 to 1")
    rationale: str = Field(description="Short explanation")


class ResolutionDecision(BaseModel):
    draft_response: str = Field(description="Draft customer response")
    needs_escalation: bool = Field(description="Whether this case should be escalated")
    escalation_reason: str = Field(description="Why escalation is needed, if any")
    confidence: float = Field(description="Confidence from 0 to 1")

**Step 7: Define graph state**

In [ ]:
class SupportState(TypedDict, total=False):
    # input
    customer_id: str
    customer_message: str

    # classification
    category: str
    intent: str
    priority: str
    sentiment: str
    classification_confidence: float
    classification_rationale: str

    # routing
    initial_route: str
    approval_decision: str
    approval_notes: str

    # retrieval
    retrieved_docs: List[Dict[str, Any]]

    # resolution
    draft_response: str
    final_response: str
    resolution_confidence: float

    # escalation
    needs_escalation: bool
    escalation_reason: str

    # bookkeeping
    status: str

**Step 8: Build helper functions**

In [ ]:
def simple_retrieve(query: str, category: str, top_k: int = 3):
    query_words = set(query.lower().split())
    scored = []

    for doc in KNOWLEDGE_BASE:
        score = 0
        if doc["category"] == category:
            score += 3

        content_words = set((doc["title"] + " " + doc["content"]).lower().split())
        overlap = len(query_words.intersection(content_words))
        score += overlap

        scored.append((score, doc))

    scored.sort(key=lambda x: x[0], reverse=True)
    docs = [doc for score, doc in scored if score > 0][:top_k]
    return docs

In [ ]:
classifier = llm.with_structured_output(TicketClassification)

def classify_with_llm(customer_message: str) -> TicketClassification:
    prompt = f"""
You are a customer support ticket classifier.

Read the ticket and classify:
- category
- intent
- priority
- sentiment
- confidence
- rationale

Ticket:
{customer_message}
"""
    return classifier.invoke(prompt)

In [ ]:
resolver = llm.with_structured_output(ResolutionDecision)

def resolve_with_llm(state: SupportState) -> ResolutionDecision:
    kb_text = "\n\n".join(
        [
            f"[{doc['id']}] {doc['title']}\nCategory: {doc['category']}\n{doc['content']}"
            for doc in state.get("retrieved_docs", [])
        ]
    )

    prompt = f"""
You are a customer support resolution assistant.

Use the ticket classification, customer message, and retrieved knowledge
to draft a helpful and safe support response.

Rules:
- If the issue involves security risk, missing verification, locked account, legal risk,
  or insufficient information, set needs_escalation=true.
- Keep the response clear, polite, and actionable.
- Do not invent policy beyond the knowledge provided.

Customer ID: {state.get('customer_id', '')}
Message: {state.get('customer_message', '')}
Category: {state.get('category', '')}
Intent: {state.get('intent', '')}
Priority: {state.get('priority', '')}
Sentiment: {state.get('sentiment', '')}
Approval notes: {state.get('approval_notes', '')}

Retrieved Knowledge:
{kb_text if kb_text else "No relevant knowledge found."}
"""
    return resolver.invoke(prompt)

**Step 9: Define graph nodes**

In [ ]:
def classify_ticket(state: SupportState) -> SupportState:
    result = classify_with_llm(state["customer_message"])

    # First routing hint
    if result.intent in ["refund_request", "payment_issue", "bug_report", "password_reset", "order_status", "account_access", "general_question"]:
        initial_route = "retrieve"
    else:
        initial_route = "escalate"

    return {
        "category": result.category,
        "intent": result.intent,
        "priority": result.priority,
        "sentiment": result.sentiment,
        "classification_confidence": result.confidence,
        "classification_rationale": result.rationale,
        "initial_route": initial_route,
        "status": "classified",
    }

In [ ]:
def approval_gate(state: SupportState) -> SupportState:
    needs_human_gate = (
        state.get("priority") in ["high", "urgent"]
        or state.get("classification_confidence", 0) < 0.70
        or state.get("category") in ["billing", "account"]
    )

    if not needs_human_gate:
        return {
            "approval_decision": "approved",
            "approval_notes": "Auto-approved: low-risk case",
            "status": "routing_approved",
        }

    human_input = interrupt({
        "stage": "classification_review",
        "message": "Review ticket classification and approve routing.",
        "ticket": state["customer_message"],
        "classification": {
            "category": state.get("category"),
            "intent": state.get("intent"),
            "priority": state.get("priority"),
            "sentiment": state.get("sentiment"),
            "confidence": state.get("classification_confidence"),
            "rationale": state.get("classification_rationale"),
            "proposed_route": state.get("initial_route"),
        },
        "expected_reply_format": {
            "decision": "approved | escalate",
            "notes": "optional reviewer notes"
        }
    })

    if isinstance(human_input, dict):
        decision = human_input.get("decision", "approved")
        notes = human_input.get("notes", "")
    else:
        decision = str(human_input)
        notes = ""

    return {
        "approval_decision": decision,
        "approval_notes": notes,
        "status": "routing_reviewed",
    }

In [ ]:
def route_after_approval(state: SupportState) -> str:
    if state.get("approval_decision") == "escalate":
        return "escalate_case"

    if state.get("initial_route") == "retrieve":
        return "retrieve_knowledge"

    return "escalate_case"

In [ ]:
def retrieve_knowledge(state: SupportState) -> SupportState:
    docs = simple_retrieve(
        query=state["customer_message"],
        category=state.get("category", "general"),
        top_k=3,
    )

    return {
        "retrieved_docs": docs,
        "status": "knowledge_retrieved",
    }

In [ ]:
def draft_resolution(state: SupportState) -> SupportState:
    decision = resolve_with_llm(state)

    return {
        "draft_response": decision.draft_response,
        "needs_escalation": decision.needs_escalation,
        "escalation_reason": decision.escalation_reason,
        "resolution_confidence": decision.confidence,
        "status": "draft_ready",
    }

In [ ]:
def final_review_gate(state: SupportState) -> SupportState:
    must_review = (
        state.get("needs_escalation", False)
        or state.get("resolution_confidence", 0) < 0.80
        or state.get("priority") in ["high", "urgent"]
        or state.get("category") in ["billing", "account"]
    )

    if not must_review:
        return {
            "final_response": state.get("draft_response", ""),
            "status": "approved_final_response",
        }

    human_input = interrupt({
        "stage": "final_response_review",
        "message": "Approve, edit, or escalate the drafted response.",
        "draft_response": state.get("draft_response"),
        "needs_escalation": state.get("needs_escalation"),
        "escalation_reason": state.get("escalation_reason"),
        "confidence": state.get("resolution_confidence"),
        "expected_reply_format": {
            "decision": "approve | edit | escalate",
            "edited_response": "required only if decision=edit",
            "notes": "optional reviewer notes"
        }
    })

    if not isinstance(human_input, dict):
        human_input = {"decision": str(human_input)}

    decision = human_input.get("decision", "approve")
    notes = human_input.get("notes", "")

    if decision == "edit":
        return {
            "final_response": human_input.get("edited_response", state.get("draft_response", "")),
            "approval_notes": (state.get("approval_notes", "") + f" | final review notes: {notes}").strip(),
            "needs_escalation": False,
            "status": "approved_final_response",
        }

    if decision == "escalate":
        return {
            "needs_escalation": True,
            "escalation_reason": notes or state.get("escalation_reason", "Human reviewer requested escalation"),
            "status": "review_requested_escalation",
        }

    return {
        "final_response": state.get("draft_response", ""),
        "approval_notes": (state.get("approval_notes", "") + f" | final review notes: {notes}").strip(),
        "status": "approved_final_response",
    }

In [ ]:
def route_resolution_or_escalation(state: SupportState) -> str:
    if state.get("needs_escalation", False):
        return "escalate_case"
    return "resolve_case"

In [ ]:
def resolve_case(state: SupportState) -> SupportState:
    return {
        "status": "resolved",
        "final_response": state.get("final_response", state.get("draft_response", "")),
    }

In [ ]:
def escalate_case(state: SupportState) -> SupportState:
    escalation_msg = (
        "Your case has been escalated to a human support specialist. "
        f"Reason: {state.get('escalation_reason', 'Needs manual review')}."
    )
    return {
        "status": "escalated",
        "final_response": escalation_msg,
    }

**Step 10: Build the LangGraph workflow**

In [ ]:
builder = StateGraph(SupportState)

builder.add_node("classify_ticket", classify_ticket)
builder.add_node("approval_gate", approval_gate)
builder.add_node("retrieve_knowledge", retrieve_knowledge)
builder.add_node("draft_resolution", draft_resolution)
builder.add_node("final_review_gate", final_review_gate)
builder.add_node("resolve_case", resolve_case)
builder.add_node("escalate_case", escalate_case)

builder.add_edge(START, "classify_ticket")
builder.add_edge("classify_ticket", "approval_gate")

builder.add_conditional_edges(
    "approval_gate",
    route_after_approval,
    {
        "retrieve_knowledge": "retrieve_knowledge",
        "escalate_case": "escalate_case",
    },
)

builder.add_edge("retrieve_knowledge", "draft_resolution")
builder.add_edge("draft_resolution", "final_review_gate")

builder.add_conditional_edges(
    "final_review_gate",
    route_resolution_or_escalation,
    {
        "resolve_case": "resolve_case",
        "escalate_case": "escalate_case",
    },
)

builder.add_edge("resolve_case", END)
builder.add_edge("escalate_case", END)

**Step 11: Add SQLite persistence**

In [ ]:
conn = sqlite3.connect("support_agent_checkpoints.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)

graph = builder.compile(checkpointer=checkpointer)

**Step 12: Run the graph with a persistent thread**

In [ ]:
thread_id = "customer-1001-session-1"
config = {"configurable": {"thread_id": thread_id}}

ticket_input = {
    "customer_id": "CUST_1001",
    "customer_message": (
        "I was charged twice for my subscription renewal and I want a refund. "
        "This is frustrating because it happened again."
    )
}

result = graph.invoke(ticket_input, config=config)
result

**Step 13: Check whether the graph paused for human approval**

In [ ]:
result

In [ ]:
{
  ...,
  "__interrupt__": [
    Interrupt(value={...})
  ]
}

In [ ]:
snapshot = graph.get_state(config)
snapshot

**Step 14: Resume after first approval gate**

In [ ]:
resume_1 = Command(
    resume={
        "decision": "approved",
        "notes": "Proceed with retrieval and draft a refund response."
    }
)

result_after_resume_1 = graph.invoke(resume_1, config=config)
result_after_resume_1

**Step 15: Resume after final review gate**

In [ ]:
resume_2 = Command(
    resume={
        "decision": "approve",
        "notes": "Response looks good."
    }
)

final_result = graph.invoke(resume_2, config=config)
final_result

In [ ]:
resume_2 = Command(
    resume={
        "decision": "edit",
        "edited_response": (
            "I'm sorry you were charged twice. We can help review the duplicate charge. "
            "Please confirm the invoice email and the transaction date so we can validate the refund request."
        ),
        "notes": "Added a clearer verification request."
    }
)

final_result = graph.invoke(resume_2, config=config)
final_result

In [ ]:
resume_2 = Command(
    resume={
        "decision": "escalate",
        "notes": "Repeated billing issue; send to billing specialist."
    }
)

final_result = graph.invoke(resume_2, config=config)
final_result

**Step 16: Print the final customer response**

In [ ]:
print("STATUS:", final_result.get("status"))
print("FINAL RESPONSE:")
print(final_result.get("final_response"))

**Step 17: Demonstrate persistence across sessions**

In [ ]:
same_config = {"configurable": {"thread_id": "customer-1001-session-1"}}
restored_state = graph.get_state(same_config)
restored_state

In [ ]:
new_config = {"configurable": {"thread_id": "customer-2002-session-1"}}

new_ticket = {
    "customer_id": "CUST_2002",
    "customer_message": (
        "My app crashes every time I try to log in after the latest update."
    )
}

new_result = graph.invoke(new_ticket, config=new_config)
new_result